# Fase 2 — Limpieza y Feature Engineering

Imputación lógica de fechas, cálculo de `inventory_age` y margen de contribución.

OBJETIVO: MAXIMIZAR LIQUIDEZ DE LA EMPRESA ELIMINANDO EL STOCK MUERTO Y OPTIMIZANDO  LA INVERSION EN LOS PRODUCTOS QUE REALMENTE GENERAN MARGEN NETO TRAS DEVOLUCIONES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet('../data/raw_master_data.parquet')
df.head()

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index
0,310417,1,Tops & Tees,Seven7,27.048,NaN,None,2025-04-25 06:18:30+00:00,2025-06-18 18:44:30+00:00,NaT,54,NaN,739.24829,0.74761,8,0,0.073047
1,219939,1,Tops & Tees,Seven7,27.048,49.0,Shipped,2020-10-30 14:05:00+00:00,NaT,NaT,2007,21.952,739.24829,0.74761,8,1,2.714920
2,219938,1,Tops & Tees,Seven7,27.048,NaN,None,2026-04-06 03:33:34+00:00,2026-04-23 15:45:34+00:00,NaT,17,NaN,739.24829,0.74761,8,0,0.022996
3,114521,1,Tops & Tees,Seven7,27.048,49.0,Complete,2024-06-19 03:55:00+00:00,NaT,NaT,679,21.952,739.24829,0.74761,8,1,0.918501
4,310419,1,Tops & Tees,Seven7,27.048,NaN,None,2020-06-10 16:31:00+00:00,NaT,NaT,2148,NaN,739.24829,0.74761,8,1,2.905654


In [2]:
df.dtypes # aqui verificamos que los tipos de datos coincidan, caso contrario se le dara el formato correcto

inventory_item_id                        int64
product_id                               int64
category                                object
brand                                   object
cost                                   float64
sale_price                             float64
status                                  object
inv_created_at             datetime64[ns, UTC]
sold_at                    datetime64[ns, UTC]
returned_at                datetime64[ns, UTC]
days_in_inventory                        int64
unit_margin                            float64
cat_avg_days_in_inv                    float64
cost_percentile_in_cat                 float64
total_sku_stock_history                  int64
is_dead_stock                            int64
inventory_age_index                    float64
dtype: object

In [3]:
# Verificamos integridad de datos, relacionado con fechas
# logica de fecha: cualquier fila donde el producto se venda antes de ser creado debe ser auditado
df[(df["inv_created_at"]>df["sold_at"]) | (df["returned_at"]<df["sold_at"])]

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index


No se muestran errores respecto a la integridad de fechas

In [4]:
# Verificamos filas donde si un producto es retornado, su status debe ser returned (cualqueir otro estado es erroneo)
df[df["returned_at"].notna() & (df["status"] != "Returned")]

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index


No se encontraron filas con errores logicos

# IMPUTACION
Desde este punto de la limpieza iniciamos con la imputacion de valores

## PRIMER OBJETIVO: `Sale_price` y `unit_margin`

Tanto sale_price y unit_margin son valores nulos para productos que nunca fueron vendidos, dado que nuestro objetivo es ver las lagunas en productos que no generan valor. Se inputaran sale_price y unit_margin con nuevas columnas `estimated_sale_price` y `estimated_unit_margin`

### Principales metodos a considerar apriori

- Inputacion mediante media de product_id
- Inputacion mediante mediana de product_id
- Casos extremos media o mediana por category

### Problemas
 - Los outliers generan sensibilidad en la media, por lo tanto sera necesario hacer un analisis de outliers e histograma (para ver la estabilidad de los precios de cada producto) y finalmente decidir el metodo estadistico correcto para su inputacion.

 #### Consideraciones
 Se considera hacer una funcion de limpieza en src/ que audite n_sales, median_price, mean_price, std_price, IQR y cv para automatizar el proceso de inputacion

In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent)) # dado que el notebook esta en notebooks/ necesitamos resolver el path
from src.limpieza import impute_inventory_pricing #importamos la funcion de limpieza para este paso del proyecto

df_imputado = impute_inventory_pricing(df) #aplicamos sobre el df completo y verificamos la imputacion
df_imputado

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin
0,310417,1,Tops & Tees,Seven7,27.048,NaN,None,2025-04-25 06:18:30+00:00,2025-06-18 18:44:30+00:00,NaT,54,NaN,739.24829,0.747610,8,0,0.073047,31.950001,CATEGORY_MEDIAN,4.902001
1,219939,1,Tops & Tees,Seven7,27.048,49.0,Shipped,2020-10-30 14:05:00+00:00,NaT,NaT,2007,21.952,739.24829,0.747610,8,1,2.714920,49.000000,REAL,21.952000
2,219938,1,Tops & Tees,Seven7,27.048,NaN,None,2026-04-06 03:33:34+00:00,2026-04-23 15:45:34+00:00,NaT,17,NaN,739.24829,0.747610,8,0,0.022996,31.950001,CATEGORY_MEDIAN,4.902001
3,114521,1,Tops & Tees,Seven7,27.048,49.0,Complete,2024-06-19 03:55:00+00:00,NaT,NaT,679,21.952,739.24829,0.747610,8,1,0.918501,49.000000,REAL,21.952000
4,310419,1,Tops & Tees,Seven7,27.048,NaN,None,2020-06-10 16:31:00+00:00,NaT,NaT,2148,NaN,739.24829,0.747610,8,1,2.905654,31.950001,CATEGORY_MEDIAN,4.902001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487611,371401,29120,Accessories,Boconi,29.484,NaN,None,2022-11-04 01:25:00+00:00,NaT,NaT,1272,NaN,732.70233,0.815054,28,1,1.736039,78.000000,PRODUCT_STABLE_MEDIAN,48.516000
487612,262837,29120,Accessories,Boconi,29.484,78.0,Processing,2020-06-22 14:27:00+00:00,NaT,NaT,2137,48.516,732.70233,0.815054,28,1,2.916601,78.000000,REAL,48.516000
487613,71065,29120,Accessories,Boconi,29.484,NaN,None,2023-04-15 07:28:00+00:00,2023-05-28 11:17:00+00:00,NaT,43,NaN,732.70233,0.815054,28,0,0.058687,78.000000,PRODUCT_STABLE_MEDIAN,48.516000
487614,42989,29120,Accessories,Boconi,29.484,78.0,Shipped,2024-01-20 10:17:00+00:00,NaT,NaT,830,48.516,732.70233,0.815054,28,1,1.132793,78.000000,REAL,48.516000


In [6]:
df_imputado[df_imputado["estimated_sale_price"].isna()] #Dado que el resultado esta vacio, todas las filas fueron imputadas

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin


In [7]:
df = df_imputado

# LIMPIEZA DE DUPLICADOS
Se hara una detección de duplicados de manera general (coincidencia total) y tambien de columnas que funcionan como PK tales como `inventory_item_id` 

## EXTRAS
- `brand` y `category` no deben tener espacios extra al inicio/final
- Uniformizar en mayusculas o minusculas 

In [8]:
df[df.duplicated(subset=['inventory_item_id'], keep=False)]

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin


# Hallazgo
Efectivamente no existe duplicados en bajo la columna `inventory_item_id`, lo que indica buena normalizacion de la base de datos y menos riesgo de problemas en PK

In [9]:
# Estandarizacion de brand y category 
df[['brand','category']] = (df[['brand','category']].apply(lambda col: col.str.replace(r'^\s+|\s+$','',regex=True)))

# se puede hacer una version mas simplificada con .strip, pero regex es mas potente para casos complejos
df

,inventory_item_id,product_id,category,brand,cost,sale_price,status,inv_created_at,sold_at,returned_at,days_in_inventory,unit_margin,cat_avg_days_in_inv,cost_percentile_in_cat,total_sku_stock_history,is_dead_stock,inventory_age_index,estimated_sale_price,imputation_source,estimated_unit_margin
0,310417,1,Tops & Tees,Seven7,27.048,NaN,None,2025-04-25 06:18:30+00:00,2025-06-18 18:44:30+00:00,NaT,54,NaN,739.24829,0.747610,8,0,0.073047,31.950001,CATEGORY_MEDIAN,4.902001
1,219939,1,Tops & Tees,Seven7,27.048,49.0,Shipped,2020-10-30 14:05:00+00:00,NaT,NaT,2007,21.952,739.24829,0.747610,8,1,2.714920,49.000000,REAL,21.952000
2,219938,1,Tops & Tees,Seven7,27.048,NaN,None,2026-04-06 03:33:34+00:00,2026-04-23 15:45:34+00:00,NaT,17,NaN,739.24829,0.747610,8,0,0.022996,31.950001,CATEGORY_MEDIAN,4.902001
3,114521,1,Tops & Tees,Seven7,27.048,49.0,Complete,2024-06-19 03:55:00+00:00,NaT,NaT,679,21.952,739.24829,0.747610,8,1,0.918501,49.000000,REAL,21.952000
4,310419,1,Tops & Tees,Seven7,27.048,NaN,None,2020-06-10 16:31:00+00:00,NaT,NaT,2148,NaN,739.24829,0.747610,8,1,2.905654,31.950001,CATEGORY_MEDIAN,4.902001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487611,371401,29120,Accessories,Boconi,29.484,NaN,None,2022-11-04 01:25:00+00:00,NaT,NaT,1272,NaN,732.70233,0.815054,28,1,1.736039,78.000000,PRODUCT_STABLE_MEDIAN,48.516000
487612,262837,29120,Accessories,Boconi,29.484,78.0,Processing,2020-06-22 14:27:00+00:00,NaT,NaT,2137,48.516,732.70233,0.815054,28,1,2.916601,78.000000,REAL,48.516000
487613,71065,29120,Accessories,Boconi,29.484,NaN,None,2023-04-15 07:28:00+00:00,2023-05-28 11:17:00+00:00,NaT,43,NaN,732.70233,0.815054,28,0,0.058687,78.000000,PRODUCT_STABLE_MEDIAN,48.516000
487614,42989,29120,Accessories,Boconi,29.484,78.0,Shipped,2024-01-20 10:17:00+00:00,NaT,NaT,830,48.516,732.70233,0.815054,28,1,1.132793,78.000000,REAL,48.516000


# FINALIZACION Y USO DE ESTRUCTURA MEDALLON
Dado que se finaliza el proceso de limpieza para la data, procedemos a guardar la data usando la arquitectura medallon (data limpia de manera local) usando un .parquet y su disponibilidad para futuro uso en `03_analisis_outliers.ipynb`

In [10]:
df.to_parquet('../data/silver_data_clean.parquet', index = False)